# VoltRelay Energy analysis

Reproduce this hackathon analysis in Colab. The data is synthetic. Upload the dataset ZIP to your Drive first and keep it private while analyzing. Do not publish rider-level records. Upload `analyze.py` from the project pack in Colab's Files panel.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import zipfile
ZIP_PATH=Path('/content/drive/MyDrive/hackathon-data.zip') # edit this path
PROJECT=Path('/content/volt_relay'); PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    for n in z.namelist():
        p=Path(n)
        if p.is_absolute() or '..' in p.parts: raise ValueError('Unsafe archive path: '+n)
    z.extractall(PROJECT)
print('Extracted',len(list(PROJECT.rglob('*.csv'))),'CSV files')


## Run the analysis
Upload `analyze.py` from this project pack to Colab Files. If the ZIP contains a nested source folder, the next cell finds the CSV folder and places it beside the script.


In [ ]:
import shutil,sys,subprocess
if Path('/content/analyze.py').exists(): shutil.copy('/content/analyze.py',PROJECT/'analyze.py')
event=next(PROJECT.rglob('swap_events.csv'))
raw=PROJECT/'raw'; raw.mkdir(exist_ok=True)
for f in event.parent.glob('*.csv'): shutil.copy2(f,raw/f.name)
script=PROJECT/'analyze.py'
if not script.exists(): raise FileNotFoundError('Upload analyze.py into Colab Files first.')
subprocess.run([sys.executable,str(script)],cwd=PROJECT,check=True)


In [ ]:
import json
s=json.loads((PROJECT/'results'/'summary.json').read_text())
for k in ['event_rows','completion_rate','technical_failure_rate','abandonment_rate','contribution_proxy_per_completed_swap_inr']: print(k,s[k])
print(s['data_quality'])


## Guardrails

- The records are simulated, not real company data.
- Descriptive associations do not prove causes.
- Unit economics exclude depreciation, labour, finance, tax and overhead.
- The constructed 30-day repeat measure is saturated; use well-defined signup cohorts and 30/60/90-day retention.
